# Phase 3 visual case review

Run Notebook 07 first. This notebook reads its saved forecast/observation rows and renders one McLean-centered before/after radar comparison for each historical case. It selects the highest-probability +30-minute forecast in each case, so we inspect the baseline where it expressed the most concern.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.impact import local_xy_from_lat_lon
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.tracking import MultiClusterTracker, RadarFrame, cluster_cells, detect_cells

catalog = json.loads((repo / 'config' / 'phase3_seed_catalog.json').read_text())
target = catalog['target']
results_path = repo / 'data' / 'phase3_smoke_results.jsonl'
if not results_path.exists():
    raise RuntimeError('Run Notebook 07 first; its saved phase3_smoke_results.jsonl is required.')
results = pd.read_json(results_path, lines=True)
for column in ['scan_time', 'forecast_time']:
    results[column] = pd.to_datetime(results[column], utc=True)
print(f"Loaded {len(results)} Phase 3 forecast/observation rows.")

In [ ]:
filename_pattern = re.compile(r'KLWX(\d{8})_(\d{6})_V\d+$')

def timestamp_from_path(path):
    match = filename_pattern.match(path.name)
    if not match:
        raise ValueError(f'Unexpected Level II filename: {path.name}')
    return datetime.strptime(''.join(match.groups()), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)

def closest_cached_path(event_id, target_time):
    paths = list((repo / 'data' / 'phase3' / event_id).glob('KLWX*_V*'))
    if not paths:
        raise RuntimeError(f'No cached scans for {event_id}; rerun Notebook 07.')
    target_time = target_time.to_pydatetime() if hasattr(target_time, 'to_pydatetime') else target_time
    return min(paths, key=lambda path: abs(timestamp_from_path(path) - target_time))

def plot_frame(ax, frame, title, *, observed):
    office_x, office_y = local_xy_from_lat_lon(
        target['latitude_deg'], target['longitude_deg'],
        frame.radar_latitude_deg, frame.radar_longitude_deg,
    )
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz,
                          shading='auto', cmap='turbo', vmin=0, vmax=65)
    circle = plt.Circle((office_x / 1_000, office_y / 1_000),
                        target['label_radius_m'] / 1_000, fill=False, color='white',
                        linewidth=1.8, linestyle='--', zorder=5)
    ax.add_patch(circle)
    ax.scatter(office_x / 1_000, office_y / 1_000, marker='*', s=175, c='#ff4d4d',
               edgecolors='white', linewidths=.8, zorder=6, label='McLean office')
    ax.set(title=title, xlim=(office_x / 1_000 - 50, office_x / 1_000 + 50),
           ylim=(office_y / 1_000 - 50, office_y / 1_000 + 50), aspect='equal',
           xlabel='km east of KLWX', ylabel='km north of KLWX')
    return image


In [ ]:
selected = (results[results['horizon_minutes'] == 30]
            .sort_values('combined_probability', ascending=False)
            .groupby('event_id', as_index=False).first())
display(selected[['event_id', 'scan_time', 'forecast_time', 'combined_probability',
                  'active_track_count', 'observed_35dbz_coverage']])

fig, axes = plt.subplots(len(selected), 2, figsize=(15, 7 * len(selected)), squeeze=False)
for row_index, (_, row) in enumerate(selected.iterrows()):
    origin_frame = load_lowest_reflectivity_sweep(closest_cached_path(row['event_id'], row['scan_time']))
    verification_frame = load_lowest_reflectivity_sweep(closest_cached_path(row['event_id'], row['forecast_time']))
    image = plot_frame(axes[row_index, 0], origin_frame,
        f"{row['event_id']} · forecast origin {origin_frame.observed_at:%H:%M UTC}\n"
        f"+30 min forecast: {row['combined_probability']:.0%} · {row['active_track_count']} active tracks",
        observed=False)
    actual = 'YES' if row['observed_35dbz_coverage'] else 'NO'
    plot_frame(axes[row_index, 1], verification_frame,
        f"Verification scan {verification_frame.observed_at:%H:%M UTC}\n"
        f"Observed ≥{target['threshold_dbz']} dBZ within 2 km of McLean: {actual}",
        observed=True)
    fig.colorbar(image, ax=axes[row_index].tolist(), label='reflectivity (dBZ)', shrink=.84)
fig.suptitle('Phase 3 visual review: McLean-centered forecast origins and later radar observations',
             fontsize=15, weight='bold', y=1.01)
fig.tight_layout()
plt.show()

print('The white dashed circle is the 2 km area used for the observed label. The red star is the McLean office.')

## Cluster-level predicted versus actual motion

The earlier before/after review stays unchanged. Use the added maps in this order:

1. **Left map — what the tracker saw and forecast.** A filled numbered icon is a real cluster at the forecast origin. Follow its arrow to the hollow icon: that is where the Kalman tracker predicted that same numbered cluster would be about 30 minutes later.
2. **Right map — what radar actually showed later.** The hollow numbered icon is repeated so you can compare it with the filled icon of the same number. The dotted line between them is the miss distance. If the filled later icon is inside the dashed circle, the actual centroid landed within the tracker’s one-standard-deviation location uncertainty.

A missing later filled icon means **unmatched**: the track may have weakened below 35 dBZ, merged, split, moved beyond the association gate, or disappeared. It is not automatically a bad forecast. These maps score **cluster-centroid motion**, not the probability that the full storm footprint hits McLean. Only clusters that enter the visible 100 × 100 km McLean-centered map receive an icon, so off-map clusters do not make the chart unreadable.

In [ ]:
TRACKING_RADIUS_M = 120_000
VIEW_HALF_WIDTH_M = 50_000

def tracking_frame(raw_frame):
    office_x, office_y = local_xy_from_lat_lon(
        target['latitude_deg'], target['longitude_deg'],
        raw_frame.radar_latitude_deg, raw_frame.radar_longitude_deg,
    )
    inside = np.hypot(raw_frame.x_m - office_x, raw_frame.y_m - office_y) <= TRACKING_RADIUS_M
    return (RadarFrame(
        raw_frame.observed_at, np.where(inside, raw_frame.reflectivity_dbz, np.nan),
        raw_frame.x_m, raw_frame.y_m, raw_frame.radar_latitude_deg, raw_frame.radar_longitude_deg,
    ), office_x, office_y)

def rebuild_track_review(event_id, scan_time, forecast_time):
    """Rebuild the Phase 3 tracker at one saved forecast origin without retaining all volumes."""
    event_paths = sorted((repo / 'data' / 'phase3' / event_id).glob('KLWX*_V*'), key=timestamp_from_path)
    origin_path = closest_cached_path(event_id, scan_time)
    verification_path = closest_cached_path(event_id, forecast_time)
    origin_time = timestamp_from_path(origin_path)
    tracker = MultiClusterTracker()
    origin_frame = origin_update = None

    for path in (path for path in event_paths if timestamp_from_path(path) <= origin_time):
        raw_frame = load_lowest_reflectivity_sweep(path)
        frame, office_x, office_y = tracking_frame(raw_frame)
        clusters = cluster_cells(
            detect_cells(frame, threshold_dbz=target['threshold_dbz'], min_gates=20),
            linkage_distance_m=15_000,
        )
        update = tracker.update(frame.observed_at, clusters)
        if path == origin_path:
            origin_frame, origin_update = raw_frame, update
        else:
            del raw_frame
        del frame, clusters
        gc.collect()

    if origin_frame is None or origin_update is None:
        raise RuntimeError(f'Could not rebuild tracks at {scan_time} for {event_id}.')
    verification_frame = load_lowest_reflectivity_sweep(verification_path)
    verification_tracking, _, _ = tracking_frame(verification_frame)
    verification_clusters = cluster_cells(
        detect_cells(verification_tracking, threshold_dbz=target['threshold_dbz'], min_gates=20),
        linkage_distance_m=15_000,
    )
    forecast_tracks = tracker.forecast(verification_frame.observed_at)
    verification_update = tracker.update(verification_frame.observed_at, verification_clusters)
    observed_by_track = {track.track_id: track.observed_cluster for track in verification_update.tracks}
    return origin_frame, verification_frame, office_x, office_y, forecast_tracks, observed_by_track

def in_mclean_view(x_m, y_m, office_x, office_y):
    return abs(x_m - office_x) <= VIEW_HALF_WIDTH_M and abs(y_m - office_y) <= VIEW_HALF_WIDTH_M

def numbered_icon(ax, x_m, y_m, track_id, color, *, filled, zorder=10):
    x_km, y_km = x_m / 1_000, y_m / 1_000
    facecolor = color if filled else 'none'
    ax.scatter(x_km, y_km, s=300, marker='o', facecolors=facecolor, edgecolors='white',
               linewidths=1.8, zorder=zorder)
    ax.text(x_km, y_km, str(track_id), color='white', ha='center', va='center',
            weight='bold', fontsize=8, zorder=zorder + 1)

movement_rows = []
for _, row in selected.iterrows():
    origin_frame, verification_frame, office_x, office_y, forecasts, observed_by_track = rebuild_track_review(
        row['event_id'], row['scan_time'], row['forecast_time'],
    )
    records = []
    for forecast in forecasts:
        start = forecast.last_observed_cluster
        observed = observed_by_track.get(forecast.track_id)
        visible = (
            in_mclean_view(start.centroid_x_m, start.centroid_y_m, office_x, office_y)
            or in_mclean_view(forecast.estimate.x_m, forecast.estimate.y_m, office_x, office_y)
            or (observed is not None and in_mclean_view(observed.centroid_x_m, observed.centroid_y_m, office_x, office_y))
        )
        if not visible:
            continue
        error_km = (np.hypot(observed.centroid_x_m - forecast.estimate.x_m,
                              observed.centroid_y_m - forecast.estimate.y_m) / 1_000
                    if observed is not None else np.nan)
        records.append((forecast, observed, error_km))
        movement_rows.append({
            'event_id': row['event_id'], 'track_id': forecast.track_id,
            'forecast_position_std_km': round(forecast.estimate.position_std_m / 1_000, 1),
            'position_error_km': round(error_km, 1) if np.isfinite(error_km) else np.nan,
            'association_at_verification': 'matched' if observed is not None else 'unmatched',
        })

    # A square map area needs a generous canvas. Constrained layout reserves space
    # for titles and the colorbar instead of squeezing annotation into the radar view.
    fig, axes = plt.subplots(1, 2, figsize=(21, 10), squeeze=False, layout='constrained')
    left, right = axes[0]
    image = plot_frame(left, origin_frame,
        f"Observed clusters · {origin_frame.observed_at:%H:%M UTC}", observed=False)
    plot_frame(right, verification_frame,
        f"Predicted versus observed · {verification_frame.observed_at:%H:%M UTC}", observed=True)

    palette = plt.get_cmap('tab20')
    for forecast, observed, error_km in records:
        color = palette((forecast.track_id - 1) % 20)
        start = forecast.last_observed_cluster
        numbered_icon(left, start.centroid_x_m, start.centroid_y_m, forecast.track_id, color, filled=True)
        left.annotate('', xy=(forecast.estimate.x_m / 1_000, forecast.estimate.y_m / 1_000),
                      xytext=(start.centroid_x_m / 1_000, start.centroid_y_m / 1_000),
                      arrowprops={'arrowstyle': '->', 'color': color, 'lw': 1.8, 'alpha': .9}, zorder=8)
        for axis in (left, right):
            forecast_circle = plt.Circle((forecast.estimate.x_m / 1_000, forecast.estimate.y_m / 1_000),
                                         forecast.estimate.position_std_m / 1_000, fill=False,
                                         color=color, linestyle='--', linewidth=1.5, alpha=.8, zorder=7)
            axis.add_patch(forecast_circle)
        numbered_icon(left, forecast.estimate.x_m, forecast.estimate.y_m, forecast.track_id, color, filled=False)
        numbered_icon(right, forecast.estimate.x_m, forecast.estimate.y_m, forecast.track_id, color, filled=False)
        if observed is not None:
            right.plot([forecast.estimate.x_m / 1_000, observed.centroid_x_m / 1_000],
                       [forecast.estimate.y_m / 1_000, observed.centroid_y_m / 1_000],
                       color=color, linestyle=':', linewidth=2, zorder=8)
            numbered_icon(right, observed.centroid_x_m, observed.centroid_y_m, forecast.track_id, color, filled=True, zorder=11)

    fig.colorbar(image, ax=axes[0].tolist(), label='reflectivity (dBZ)', shrink=.88, pad=.02)
    fig.text(.5, .01, f"{len(records)} numbered tracks visible near McLean out of {len(forecasts)} active tracks · filled = observed · hollow = predicted · dashed = 1σ uncertainty · dotted = miss distance",
             ha='center', fontsize=11, weight='bold')
    plt.show()
    plt.close(fig)
    del origin_frame, verification_frame
    gc.collect()

movement_summary = pd.DataFrame(movement_rows, columns=[
    'event_id', 'track_id', 'forecast_position_std_km', 'position_error_km', 'association_at_verification',
])
if movement_summary.empty:
    print('No active tracks entered the displayed McLean-centered window for these selected cases.')
else:
    # Start with a plain-English case summary; the detailed per-track table follows.
    case_summary = []
    for event_id, event_rows in movement_summary.groupby('event_id'):
        matched = event_rows.dropna(subset=['position_error_km'])
        covered = (matched['position_error_km'] <= matched['forecast_position_std_km']).sum()
        case_summary.append({
            'event_id': event_id,
            'visible_tracks': len(event_rows),
            'matched_tracks': len(matched),
            'unmatched_tracks': len(event_rows) - len(matched),
            'median_error_km_matched_only': round(matched['position_error_km'].median(), 1) if len(matched) else np.nan,
            'matched_inside_1sigma': f'{covered}/{len(matched)}' if len(matched) else 'n/a',
        })
    print('How to read this: smaller median error is better. A matched actual centroid inside its 1σ circle means the uncertainty estimate covered it; it does not mean the full storm shape was predicted exactly.')
    display(pd.DataFrame(case_summary))
    print('Detailed track diagnostics')
    display(movement_summary.sort_values(['event_id', 'track_id']))
print('Filled numbered icon = observed cluster. Hollow numbered icon = predicted centroid. Dashed circle = 1σ location uncertainty. Dotted line = prediction-to-observation error.')